# 01 — Dentro de un objeto Seurat

**Taller de célula única CIAD**

Antes de analizar nada, conviene saber qué es lo que tenemos en las manos.

Un objeto Seurat es un contenedor. Entran los counts, y cada resultado que
calculamos — métricas de calidad, clusters, coordenadas UMAP, etiquetas de tipo
celular — se guarda de vuelta en el mismo objeto en lugar de quedar disperso en
variables sueltas. Eso es cómodo, pero significa que el objeto se va llenando de
cosas cuya ubicación no es obvia.

Este notebook lo abre. Los mismos datos que el resto del taller: 2,700 PBMCs de
10x Genomics.

**Qué deberías poder hacer al terminar**

- decir qué es un assay, una layer y una reduction
- extraer los counts, la metadata, los nombres de células, los nombres de genes
- agregar tu propia columna de metadata
- hacer un subset del objeto sin romperlo
- encontrar dónde quedó guardado un resultado, sin adivinar

Unos 30 minutos. Ejecuta las celdas con *Shift + Enter*.

## Preparación

In [ ]:
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup.R")

## 1. Antes del objeto: la matriz

Todo empieza como una matriz de counts. Genes en las filas, células en las
columnas, y cada entrada es el número de moléculas de ese gen detectadas en esa
célula.

In [ ]:
url <- "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"

download.file(url, "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

counts <- Read10X(data.dir = "filtered_gene_bc_matrices/hg19")

dim(counts)

32,738 genes y 2,700 células. Veamos una esquina de la matriz — cinco
genes, cinco células.

In [ ]:
counts[1:5, 1:5]

Todo ceros, y no es un error. Si elegimos genes que sí se expresan,
el panorama cambia.

In [ ]:
counts[c("CD3D", "CD3E", "MS4A1", "CD14", "LYZ"), 1:5]

### Por qué los puntos

Fíjate en el `.` en lugar del `0`.

Esta no es una matriz común. Es una matriz **dispersa** — clase `dgCMatrix` — que
guarda únicamente las entradas distintas de cero. Un punto significa "aquí no hay
nada guardado", es decir, cero.

Esto importa más de lo que parece. La mayoría de las entradas en datos de célula
única son cero, y guardarlas todas sería un desperdicio hasta el punto de ser
imposible.

In [ ]:
cat("class:", class(counts), "\n\n")

zeros <- 1 - length(counts@x) / (nrow(counts) * ncol(counts))
cat(sprintf("entries that are zero: %.1f%%\n\n", 100 * zeros))

cat("sparse :", format(object.size(counts), units = "MB"), "\n")
cat("dense  :", format(object.size(as.matrix(counts[1:2000, ])) * (nrow(counts) / 2000),
                      units = "MB"), "(estimated)\n")

## 2. Creando el objeto

`CreateSeuratObject()` envuelve esa matriz en algo capaz de cargar resultados.

In [ ]:
pbmc <- CreateSeuratObject(
  counts       = counts,
  project      = "pbmc3k",
  min.cells    = 3,
  min.features = 200
)

pbmc

Lee ese resumen línea por línea. Dice: un **assay**, `RNA`, que es el
**activo**; 13,714 features; 2,700 samples; una **layer**, llamada `counts`.

Hay menos genes que al inicio — `min.cells = 3` eliminó los genes detectados en
casi ninguna célula.

### El vocabulario

| término | qué significa |
|---|---|
| **assay** | un tipo de medición sobre las mismas células — aquí `RNA`. Un experimento CITE-seq tendría además `ADT` para proteínas de superficie. La integración agrega un assay corregido junto al original. |
| **layer** | una versión de la matriz dentro de un assay: `counts` es cruda, `data` normalizada, `scale.data` centrada y escalada. Mismas células, mismos genes, números distintos. (En Seurat v4 se llamaban *slots*.) |
| **metadata** | una fila por célula, una columna por cada cosa que sabemos de ella: métricas de calidad, muestra de origen, cluster, tipo celular. |
| **reduction** | una representación en pocas dimensiones — PCA, UMAP. Una fila por célula, unas pocas columnas. |
| **identity** | la etiqueta actual de la célula. Lo que devuelve `Idents()`. Es lo que usan los gráficos para colorear y lo que compara la expresión diferencial. |

El resto del notebook trata de dónde vive cada una de esas cosas y cómo llegar a
ellas.

## 3. Assays y layers

Un assay es una caja con nombre. Los listamos, y vemos cuál está activo.

In [ ]:
cat("assays        :", Assays(pbmc), "\n")
cat("active assay  :", DefaultAssay(pbmc), "\n")
cat("layers in RNA :", Layers(pbmc[["RNA"]]), "\n")

Una sola layer por ahora, porque no hemos normalizado. Al hacerlo
aparece una segunda — los counts crudos no se reemplazan.

In [ ]:
pbmc <- NormalizeData(pbmc, verbose = FALSE)

Layers(pbmc[["RNA"]])

### Sacar la matriz

`LayerData()` es la función de acceso. Se le indica el assay y la layer que
quieres.

Meter la mano con `@` también funciona, y lo verás en código antiguo, pero la
estructura interna cambió entre Seurat v4 y v5 — la función de acceso no. Usa la
función de acceso.

In [ ]:
raw  <- LayerData(pbmc, assay = "RNA", layer = "counts")
norm <- LayerData(pbmc, assay = "RNA", layer = "data")

genes <- c("CD3D", "MS4A1", "CD14", "LYZ", "PPBP")
cells <- colnames(pbmc)[1:5]

cat("--- counts (raw molecules) ---\n")
print(raw[genes, cells])

cat("\n--- data (log-normalised) ---\n")
print(round(norm[genes, cells], 2))

Mismos genes, mismas células, dos layers. La layer cruda tiene
números enteros — moléculas contadas. La normalizada tiene decimales, corregidos
según cuánto RNA aportó cada célula y luego transformados con logaritmo.

Los ceros siguen siendo cero. La normalización no inventa expresión.

## 4. Metadata

Una fila por célula. Aquí vive todo lo que sabemos *acerca de* las células, a
diferencia de lo que expresan.

In [ ]:
dim(pbmc@meta.data)

head(pbmc@meta.data, 5)

Ya hay tres columnas, y no agregamos ninguna:

- `orig.ident` — el nombre de `project` que le pasamos
- `nCount_RNA` — total de moléculas en la célula
- `nFeature_RNA` — genes detectados en la célula

Las dos últimas las calculó `CreateSeuratObject()` porque se necesitan muy
seguido.

### Dos formas de leer una columna

`pbmc$nombre` es la forma corta. `pbmc[["nombre"]]` devuelve un data frame en
lugar de un vector — a veces es lo que quieres, normalmente no.

In [ ]:
summary(pbmc$nFeature_RNA)

cat("\nvector      :", class(pbmc$nCount_RNA), "\n")
cat("data frame  :", class(pbmc[["nCount_RNA"]]), "\n")

### Agregar una columna

Asigna con `$` un valor por célula, en el orden en que están las células.

In [ ]:
pbmc$percent.mt <- PercentageFeatureSet(pbmc, pattern = "^MT-")

# anything of the right length works — here, a crude size label
pbmc$size <- ifelse(pbmc$nCount_RNA > median(pbmc$nCount_RNA), "large", "small")

head(pbmc@meta.data, 3)

table(pbmc$size)

### ✏️ Ejercicio 1

Agrega una columna llamada `high.mito` que sea `TRUE` cuando el porcentaje
mitocondrial de la célula sea mayor a 5, y `FALSE` en caso contrario. Después
cuenta cuántas células son.

Completa el espacio en blanco:

In [ ]:
# pbmc$high.mito <- ______

table(pbmc$high.mito)

## 5. Células y genes

Los nombres, no los números, son la forma de referirse a las cosas.

In [ ]:
cat("cells:", ncol(pbmc), " genes:", nrow(pbmc), "\n\n")

cat("first 3 cell names:\n"); print(head(Cells(pbmc), 3))
cat("\nfirst 3 gene names:\n"); print(head(Features(pbmc), 3))

Los nombres de las células son los barcodes de 10x — la etiqueta de
DNA que identificó la gota. Son los nombres de fila de la metadata y los nombres
de columna de la matriz, y Seurat mantiene esa correspondencia por ti.

`colnames()` y `rownames()` también funcionan y significan lo mismo.

In [ ]:
identical(Cells(pbmc), colnames(pbmc))
identical(Features(pbmc), rownames(pbmc))

## 6. Identities

La identity es la etiqueta actual de la célula — con lo que colorean los
gráficos, y lo que usa la expresión diferencial para agrupar.

Ahora mismo todas las células tienen la misma, porque no hemos hecho clustering.

In [ ]:
head(Idents(pbmc), 3)

table(Idents(pbmc))

Puedes asignar las identities desde cualquier columna de metadata, y
devolverlas. Vale la pena practicarlo, porque es la forma más común en que uno se
confunde a sí mismo más adelante.

In [ ]:
Idents(pbmc) <- "size"          # use the column we invented
table(Idents(pbmc))

# keep a copy before overwriting — identities are easy to lose
pbmc$my_grouping <- Idents(pbmc)

Idents(pbmc) <- "orig.ident"    # and back
table(Idents(pbmc))

## 7. Dónde van los resultados

Para mostrar reductions y grafos necesitamos haber calculado algunos. Esta celda
ejecuta el pipeline estándar de forma rápida y silenciosa — el notebook 02 es
donde explicamos qué hace cada paso y por qué. Por ahora, observa qué le agrega al
objeto.

In [ ]:
pbmc <- FindVariableFeatures(pbmc, verbose = FALSE)
pbmc <- ScaleData(pbmc, verbose = FALSE)
pbmc <- RunPCA(pbmc, verbose = FALSE)
pbmc <- FindNeighbors(pbmc, dims = 1:10, verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)
pbmc <- RunUMAP(pbmc, dims = 1:10, verbose = FALSE)

pbmc

Compara ese resumen con el de la sección 2. El objeto ahora reporta
tres layers y dos reductions.

Todo quedó dentro del objeto. Nada se devolvió a una variable suelta.

In [ ]:
cat("layers     :", Layers(pbmc[["RNA"]]), "\n")
cat("reductions :", Reductions(pbmc), "\n")
cat("graphs     :", Graphs(pbmc), "\n")
cat("metadata   :", paste(colnames(pbmc@meta.data), collapse = ", "), "\n")

Nota que `seurat_clusters` apareció solo en la metadata —
`FindClusters()` lo escribe ahí, y además lo establece como la identity.

### Reductions

Una reduction es una matriz: una fila por célula, una columna por dimensión.
`Embeddings()` obtiene las coordenadas de las células.

In [ ]:
pca  <- Embeddings(pbmc, reduction = "pca")
umap <- Embeddings(pbmc, reduction = "umap")

cat("pca  :", nrow(pca),  "cells x", ncol(pca),  "components\n")
cat("umap :", nrow(umap), "cells x", ncol(umap), "dimensions\n\n")

round(umap[1:5, ], 3)

Esos dos números por célula son todo el gráfico UMAP. `DimPlot()` los
dibuja, pero no hay nada mágico debajo — podrías graficarlos tú.

In [ ]:
p1 <- DimPlot(pbmc, reduction = "umap") + ggtitle("DimPlot")

df <- as.data.frame(umap)
colnames(df) <- c("dim1", "dim2")   # whatever Seurat named them
df$cluster <- pbmc$seurat_clusters

p2 <- ggplot(df, aes(dim1, dim2, colour = cluster)) +
  geom_point(size = 0.3) +
  ggtitle("the same numbers, plotted by hand")

p1 + p2

`Loadings()` obtiene la otra mitad de un PCA: cuánto contribuye cada
**gen** a cada componente.

In [ ]:
loadings <- Loadings(pbmc, reduction = "pca")

cat(nrow(loadings), "genes x", ncol(loadings), "components\n\n")
round(loadings[1:5, 1:3], 3)

### ✏️ Ejercicio 2

¿Qué cinco genes tienen más peso sobre el PC 1, en cualquier dirección?

Completa el espacio en blanco — buscas los valores más grandes en tamaño
absoluto:

In [ ]:
pc1 <- Loadings(pbmc, reduction = "pca")[, 1]

# head(sort(______, decreasing = TRUE), 5)

## 8. Subsetting

`subset()` toma células, genes, o ambos, y mantiene todo consistente — metadata,
layers y reductions se recortan juntos.

In [ ]:
b_cells <- subset(pbmc, subset = seurat_clusters == 3)

cat("original :", ncol(pbmc),    "cells\n")
cat("subset   :", ncol(b_cells), "cells\n\n")

# the metadata came along
cat("metadata rows:", nrow(b_cells@meta.data), "\n")
# so did the UMAP
cat("umap rows    :", nrow(Embeddings(b_cells, "umap")), "\n")

Puedes hacer subset con cualquier columna de metadata, con la
expresión de un gen, o con la identity.

In [ ]:
# by expression
cd3_pos <- subset(pbmc, subset = CD3E > 1)
cat("CD3E-positive cells:", ncol(cd3_pos), "\n")

# by identity — several clusters at once
few <- subset(pbmc, idents = c(0, 1, 2))
cat("clusters 0,1,2     :", ncol(few), "cells\n")

# by gene, keeping all cells
small <- subset(pbmc, features = VariableFeatures(pbmc)[1:100])
cat("100 genes          :", nrow(small), "genes,", ncol(small), "cells\n")

### ✏️ Ejercicio 3

Crea un objeto que contenga solo las células que **no** están en el cluster 3, y
confirma que los números suman el original.

Completa el espacio en blanco:

In [ ]:
# not_b <- subset(pbmc, subset = ______)

# ncol(not_b) + ncol(b_cells) == ncol(pbmc)

## 9. Un mapa del objeto

Dónde buscar cada cosa, en un solo lugar.

| lo que quieres | cómo obtenerlo |
|---|---|
| los counts crudos | `LayerData(obj, assay = "RNA", layer = "counts")` |
| los valores normalizados | `LayerData(obj, assay = "RNA", layer = "data")` |
| toda la metadata | `obj@meta.data` |
| una columna de metadata | `obj$columna` |
| nombres de células | `Cells(obj)` o `colnames(obj)` |
| nombres de genes | `Features(obj)` o `rownames(obj)` |
| etiquetas actuales | `Idents(obj)` |
| qué assays existen | `Assays(obj)` |
| qué layers existen | `Layers(obj[["RNA"]])` |
| qué reductions existen | `Reductions(obj)` |
| coordenadas UMAP o PCA | `Embeddings(obj, reduction = "umap")` |
| contribución de genes a los PCs | `Loadings(obj, reduction = "pca")` |
| genes variables | `VariableFeatures(obj)` |
| cuántas células / genes | `ncol(obj)` / `nrow(obj)` |

Dos hábitos que vale la pena conservar:

- **Usa las funciones de acceso, no `@`.** La estructura interna cambió entre v4 y
  v5 y puede volver a cambiar. `LayerData()` no.
- **Pon en la metadata todo lo que te importe.** `Idents()` es un solo vector y se
  sobrescribe constantemente. Una columna de metadata sobrevive.

Sigue: el notebook 02, donde hacemos el análisis de verdad — control de calidad,
clustering, y nombrar los tipos celulares.

---

### Respuestas

<details>
<summary>Haz clic para desplegar</summary>

**Ejercicio 1**

```r
pbmc$high.mito <- pbmc$percent.mt > 5
```

**Ejercicio 2**

```r
head(sort(abs(pc1), decreasing = TRUE), 5)
```

`abs()` es el punto clave: un loading negativo grande importa tanto como uno
positivo grande. El signo solo indica hacia qué extremo del componente empuja el
gen.

**Ejercicio 3**

```r
not_b <- subset(pbmc, subset = seurat_clusters != 3)
ncol(not_b) + ncol(b_cells) == ncol(pbmc)
```

</details>